<a href="https://colab.research.google.com/github/KadleczPaul/SQL-fundamentals-DataCamp/blob/main/Analiza_Comenzi_VIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import pandas as pd
import duckdb

if 'con' not in locals():
    con = duckdb.connect(database=':memory:')

# Tabel 1: Clienți
date_clienti = pd.DataFrame({
    'id_client': [1, 2, 3, 4, 5],
    'nume': ['Ana', 'Bogdan', 'Cristi', 'Diana', 'Elena'],
    'oras': ['Bucuresti', 'Cluj', 'Bucuresti', 'Timisoara', 'Cluj'],
    'id_recomandat_de': [0, 1, 1, 2, 0]
})

# Tabel 2: Comenzi Trimestrul 1
date_q1 = pd.DataFrame({
    'id_comanda': [101, 102, 103],
    'id_client': [1, 2, 2],
    'suma': [150, 200, 50]
})

# Tabel 3: Comenzi Trimestrul 2
date_q2 = pd.DataFrame({
    'id_comanda': [104, 105, 106],
    'id_client': [1, 3, 4],
    'suma': [300, 400, 100]
})

try:
    con.register('clienti', date_clienti)
    con.register('comenzi_q1', date_q1)
    con.register('comenzi_q2', date_q2)
    print("Baza de date este pregătită! Tabele: clienti, comenzi_q1, comenzi_q2")
except Exception:
    print("Tabelele există deja!")

Baza de date este pregătită! Tabele: clienti, comenzi_q1, comenzi_q2


In [62]:
query = """
select * from clienti
"""
con.execute(query).df()

,id_client,nume,oras,id_recomandat_de
0,1,Ana,Bucuresti,0
1,2,Bogdan,Cluj,1
2,3,Cristi,Bucuresti,1
3,4,Diana,Timisoara,2
4,5,Elena,Cluj,0


In [63]:
query = """
select * from comenzi_q1
"""
con.execute(query).df()

,id_comanda,id_client,suma
0,101,1,150
1,102,2,200
2,103,2,50


In [64]:
query = """
select * from comenzi_q2
"""
con.execute(query).df()

,id_comanda,id_client,suma
0,104,1,300
1,105,3,400
2,106,4,100


In [84]:
query = """
with toate_comenzile as (
  Select * from comenzi_q1
  union all
  select * from comenzi_q2
),
tabel_vip as(
  select id_client, sum(suma) as total_cheltuit
  from toate_comenzile
  group by id_client
  having sum(suma) > 150
)
select c.nume, c.oras, v.total_cheltuit, p.nume as recomandat_de
from clienti as c
join tabel_vip as v
on c.id_client = v.id_client
left join clienti as p
on c.id_recomandat_de = p.id_client
order by total_cheltuit DESC

"""
con.execute(query).df()

,nume,oras,total_cheltuit,recomandat_de
0,Ana,Bucuresti,450.0,None
1,Cristi,Bucuresti,400.0,Ana
2,Bogdan,Cluj,250.0,Ana
